In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import logging
import os

#  Logging setup 
logging.basicConfig(level=logging.INFO, filename="course_scraper.log",
                    format="%(asctime)s:%(levelname)s:%(message)s")


# PART 1: Web Scraping

urls = [
    "https://syntaxandscript.com/free-online-ai-courses/"
]

def fetch_page(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        logging.info(f"Fetched {url} successfully")
        return resp.text
    except Exception as e:
        logging.error(f"Failed to fetch {url}: {e}")
        return None

def parse_courses(html):
    courses = []
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    if table:
        for li in table.select("tr"):
            cells = li.find_all("td")
            if len(cells) >= 2:
                name = cells[0].get_text(strip=True)
                provider = cells[1].get_text(strip=True)
                prereq = cells[3].get_text(strip=True) if len(cells) > 3 else ""
                duration = cells[-1].get_text(strip=True)
                courses.append({
                    "Course Name": name,
                    "Provider": provider,
                    "Prerequisites": prereq,
                    "Duration": duration
                })
    return courses

web_courses = []
for url in urls:
    html = fetch_page(url)
    if html:
        web_courses.extend(parse_courses(html))

logging.info(f"Extracted {len(web_courses)} courses from websites")
df_web = pd.DataFrame(web_courses)


# PART 2: Save to Excel

excel_filename = "ai_courses3.xlsx"
if os.path.exists(excel_filename):
    os.remove(excel_filename)

df_web.to_excel(excel_filename, index=False)
logging.info(f"Saved Excel file: {excel_filename}")
print(f"Data saved to Excel: {excel_filename}")

# PART 3: Upload to Google Sheets

def save_df_to_gsheet(df, sheet_name):
    scope = ["https://spreadsheets.google.com/feeds",
             "https://www.googleapis.com/auth/drive"]
    creds = ServiceAccountCredentials.from_json_keyfile_name(r'C:\Users\EXAM LOGIN\Downloads\utopian-surface-481509-u8-2247f7780d8b.json', scope)
    client = gspread.authorize(creds)

    # Open existing sheet (shared with service account) instead of creating
    try:
        sheet = client.open(sheet_name).sheet1
        sheet.clear()
    except gspread.SpreadsheetNotFound:
        sheet = client.create(sheet_name).sheet1

    sheet.update([df.columns.values.tolist()] + df.values.tolist())
    logging.info(f"Saved data to Google Sheet: {sheet_name}")

# Upload web courses
if not df_web.empty:
    save_df_to_gsheet(df_web, "AI Courses - Web")

print("Data uploaded to Google Sheets successfully!")

Data saved to Excel: ai_courses3.xlsx


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\EXAM LOGIN\\Downloads\\utopian-surface-481509-u8-2247f7780d8b.json'

In [2]:
pip install gspread


   ------ --------------------------------- 1/6 [oauthlib]
   ------ --------------------------------- 1/6 [oauthlib]
   ------ --------------------------------- 1/6 [oauthlib]
   ------ --------------------------------- 1/6 [oauthlib]
   -------------------- ------------------- 3/6 [google-auth]
   -------------------- ------------------- 3/6 [google-auth]
   -------------------- ------------------- 3/6 [google-auth]
   -------------------- ------------------- 3/6 [google-auth]
   -------------------------- ------------- 4/6 [google-auth-oauthlib]
   --------------------------------- ------ 5/6 [gspread]
   ---------------------------------------- 6/6 [gspread]

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install oauth2client


   -------------------- ------------------- 1/2 [oauth2client]
   ---------------------------------------- 2/2 [oauth2client]

Note: you may need to restart the kernel to use updated packages.
